# PyCRMKit 1.2.0 - 02. Sales Pipelines & Lead Conversion
Ce notebook explore le moteur de vente et de gestion commerciale :
- **Leads** : capture, qualification et métadonnées
- **Pipelines commerciaux** : étapes ordonnées (`Stage`), probabilités de gain et étapes terminales (`WON` / `LOST`)
- **Transitions de stade** : validation stricte et rejet des sauts d'étapes interdits (`InvalidStageTransition`)
- **Conversion de Lead** : transformation idempotente en Opportunité, Contact et Organisation
- **Opportunities** : gestion financière (`Money`), montants estimés et valeur pondérée du pipe

## 1. Initialisation du CRM & Contexte Commercial

In [7]:
import sys
from pathlib import Path
from decimal import Decimal

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pycrmkit import CRM, __version__
from pycrmkit.contacts import ContactEmail
from pycrmkit.pipelines import Stage, StageOutcome, StageTransition, InvalidStageTransition

crm = CRM.memory().with_context(
    actor_id="commercial-b2b",
    correlation_id="sales-flow-session-002"
)

print(f"✅ PyCRMKit version : {__version__}")

✅ PyCRMKit version : 1.2.0


## 2. Capture et Qualification de Lead
Un prospect montre un intérêt commercial via une campagne ou un formulaire entrant.

In [8]:
org = crm.organizations.create(legal_name="AeroDynamics Global")
contact = crm.contacts.create(
    first_name="Alan",
    last_name="Turing",
    emails=(ContactEmail("alan.turing@aerodynamics.com", is_primary=True),),
)

lead = crm.leads.create(
    contact_id=contact.id,
    organization_id=org.id,
    source="conference_inbound",
)
print(f"🎯 Lead capturé : {lead.id} | Statut initial : {lead.status.value}")

# Qualification du prospect
qualified_lead = crm.leads.qualify(lead.id)
print(f"⭐ Lead qualifié avec succès ! Statut : {qualified_lead.status.value}")

🎯 Lead capturé : befe04f9-0dd2-4bba-b3a6-b6b9c7b69b03 | Statut initial : new
⭐ Lead qualifié avec succès ! Statut : qualified


## 3. Définition d'un Pipeline Commercial
Configuration d'un cycle de vente structuré avec des étapes obligatoires et des probabilités associées.

In [9]:
stages = (
    Stage("discovery", "Discovery & Cadrage", 0, Decimal("0.20")),
    Stage("demo", "Démonstration Produit", 1, Decimal("0.50")),
    Stage("negotiation", "Négociation & Devis", 2, Decimal("0.80")),
    Stage("won", "Gagné (Closing)", 3, terminal=True, outcome=StageOutcome.WON),
    Stage("lost", "Perdu / Abandon", 4, terminal=True, outcome=StageOutcome.LOST),
)

# Seules les transitions suivantes sont autorisées par le modèle
transitions = (
    StageTransition("discovery", "demo"),
    StageTransition("demo", "negotiation"),
    StageTransition("negotiation", "won"),
    StageTransition("negotiation", "lost"),
    StageTransition("discovery", "lost"),
    StageTransition("demo", "lost"),
)

pipeline = crm.pipelines.define(
    id="enterprise-sales",
    name="Vente Entreprise B2B",
    stages=stages,
    transitions=transitions,
)

print(f"📈 Pipeline configuré : {pipeline.name} (Étape initiale : {pipeline.initial_stage.name})")

📈 Pipeline configuré : Vente Entreprise B2B (Étape initiale : Discovery & Cadrage)


## 4. Conversion Idempotente du Lead en Opportunité
La conversion crée une opportunité rattachée et conserve l'historique du lead.

In [10]:
opportunity = crm.leads.convert(
    lead_id=qualified_lead.id,
    name="Déploiement Plateforme Navale B2B",
    estimated_value=Decimal("120000.00"),
    currency="EUR",
    pipeline_id=pipeline.id,
    idempotency_key="conversion-aero-turing-2026",
)

print(f"💼 Opportunité créée : {opportunity.name}")
print(f"   Montant : {opportunity.estimated_value} {opportunity.currency}")
print(f"   Stade initial : {opportunity.stage_id} (Probabilité : {opportunity.probability * 100:.0f}%)")

# Test d'idempotence : rejouer l'appel avec la même clé renvoie la même entité
retry_opp = crm.leads.convert(
    lead_id=qualified_lead.id,
    name="Déploiement Plateforme Navale B2B",
    estimated_value=Decimal("120000.00"),
    currency="EUR",
    pipeline_id=pipeline.id,
    idempotency_key="conversion-aero-turing-2026",
)
assert retry_opp.id == opportunity.id
print("🛡️ Idempotence validée : la même opportunité est restituée sans duplication.")

💼 Opportunité créée : Déploiement Plateforme Navale B2B
   Montant : 120000.00 EUR
   Stade initial : discovery (Probabilité : 20%)
🛡️ Idempotence validée : la même opportunité est restituée sans duplication.


## 5. Avancement dans le Pipeline & Gardes de Transition
Nous avançons dans le pipe et vérifions qu'un retour arrière non déclaré déclenche bien `InvalidStageTransition`.

In [11]:
# Avancement vers la Démonstration
opportunity = crm.opportunities.move(opportunity.id, to="demo")
print(f"➡️ Étape : {opportunity.stage_id} | Probabilité : {opportunity.probability * 100:.0f}%")

# Avancement vers la Négociation
opportunity = crm.opportunities.move(opportunity.id, to="negotiation")
print(f"➡️ Étape : {opportunity.stage_id} | Probabilité : {opportunity.probability * 100:.0f}%")

# Tentative de transition invalide (négociation -> discovery n'est pas autorisée)
try:
    crm.opportunities.move(opportunity.id, to="discovery")
    print("❌ ERREUR : La transition interdite a été autorisée !")
except InvalidStageTransition as exc:
    print(f"🛡️ Garde active : Transition illégale rejetée avec succès : {exc}")

# Closing : Gagné !
opportunity = crm.opportunities.move(opportunity.id, to="won")
print(f"🎉 Opportunité conclue : {opportunity.stage_id} | Statut final : {opportunity.status.value}")
print(f"   Probabilité finale : {opportunity.probability * 100:.0f}%")

➡️ Étape : demo | Probabilité : 50%
➡️ Étape : negotiation | Probabilité : 80%
🛡️ Garde active : Transition illégale rejetée avec succès : invalid stage transition from negotiation to discovery: transition is not declared by the pipeline
🎉 Opportunité conclue : won | Statut final : won
   Probabilité finale : 100%


## 6. Assertions de Validation du Notebook

In [12]:
assert qualified_lead.status.value == "qualified"
assert pipeline.id == "enterprise-sales"
assert opportunity.status.value == "won"
assert opportunity.probability == Decimal("1")
assert opportunity.estimated_value == Decimal("120000.00")

print("✨ TOUS LES TESTS DU NOTEBOOK 02 (SALES & CONVERSION) SONT VALIDÉS AVEC SUCCÈS !")

✨ TOUS LES TESTS DU NOTEBOOK 02 (SALES & CONVERSION) SONT VALIDÉS AVEC SUCCÈS !
